<a href="https://colab.research.google.com/github/trinhtattran/RAGassistant/blob/main/New_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive #Section 1
drive.mount('/content/drive')  # prompts for authorisation

# install the LangChain ecosystem and other tools
!pip install pypdf
!pip install -U langchain-core langchain-community langchain-text-splitters \
               sentence-transformers chromadb gradio pypdf

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
DATA_DIR = "/content/drive/MyDrive/RAG/data"

In [3]:
from pathlib import Path #section 2
from langchain_community.document_loaders import PyPDFLoader

def load_pdfs(data_dir: str):
    pdf_paths = list(Path(data_dir).glob('*.pdf'))
    documents = []
    for path in pdf_paths:
        loader = PyPDFLoader(str(path))
        pages = loader.load()  # one Document per page
        for page in pages:
            page.metadata['source'] = path.name
        documents.extend(pages)
    return documents

raw_docs = load_pdfs(DATA_DIR)
print(f"Loaded {len(raw_docs)} pages from {len(set(d.metadata['source'] for d in raw_docs))} PDFs")

Loaded 139 pages from 17 PDFs


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter #section 3

def chunk_documents(documents):
    # Larger chunks with overlap for better context
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    return splitter.split_documents(documents)

chunks = chunk_documents(raw_docs)
print(f"Created {len(chunks)} chunks from {len(raw_docs)} pages")

Created 714 chunks from 139 pages


In [12]:
from sentence_transformers import SentenceTransformer #section 4
import chromadb

def create_vector_db(chunks, persist_directory="/content/drive/MyDrive/RAG/chroma_db"):
    texts = [doc.page_content for doc in chunks]
    metadatas = [doc.metadata for doc in chunks]
    ids = [f"chunk_{i}" for i in range(len(chunks))]

    # Load the embedding model
    embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=True)

    # Create a persistent Chroma client and collection
    client = chromadb.PersistentClient(path=persist_directory)
    collection = client.get_or_create_collection(name="parkinsons_docs")
    collection.add(
        ids=ids,
        embeddings=embeddings.tolist(),
        documents=texts,
        metadatas=metadatas
    )
    return client, collection, embed_model

# Build the vector database once
client, collection, embed_model = create_vector_db(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

In [13]:
def retrieve(query: str, top_k: int = 8, model=embed_model, coll=collection): #section 5
    """
    Embed the query, search the vector store and return the top‑K documents and metadata.
    """
    query_embedding = model.encode([query])
    results = coll.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas"]
    )
    return results['documents'][0], results['metadatas'][0]

# Test retrieval to inspect the context
docs, metas = retrieve("What are the motor symptoms of Parkinson's disease?", top_k=8)
for d, m in zip(docs, metas):
    print(m['source'], m.get('page', 'unknown'), "→", d[:120], "…")

doc (14).pdf 9 → Mid-stage
Parkinson disease
Late-stage
Parkinson disease
Depression
Apathy
Orthostatic
hypotension
DementiaUrinary
sympt …
doc (15).pdf 2 → Natural History   
and Clinical Course
Motor symptoms of slowness and tremor tend to 
be asymmetrical. Eventually, bilat …
doc (14).pdf 7 → is commonly defined by an age of onset <45 years and 
>10% of those individuals have a genetic basis, and the 
proportio …
doc (2).pdf 0 → W hat are the symptoms of atypical Parkinsonian disorders?
Like classic Parkinson’s disease, atypical Parkinsonian disor …
doc (14).pdf 7 → Diagnosis, screening and prevention
Clinical diagnosis and natural history
Parkinson disease is clinically defined by th …
doc (14).pdf 9 → Dysphagia
Motor symptoms
Non-motor symptoms
Postural instability
and gait disorder
Figure 5 | Clinical symptoms associat …
doc (12).pdf 2 → of this exercise is two-fold. First, to conﬁrm the presence of bradykinesia and
at least one other motor manifestation,  …
doc (6).pdf 0 → Copyr

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM #section 6

# Load a larger generation model (requires more RAM)
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-large')
model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-large')

def answer_question(query: str, top_k: int = 8, max_new_tokens: int = 256):
    # Retrieve relevant chunks
    docs, metas = retrieve(query, top_k=top_k)
    # Build the context string with citation markers
    context_lines = []
    citations = []
    for i, (doc_text, meta) in enumerate(zip(docs, metas)):
        label = f"[{i+1}]"
        context_lines.append(
            f"{label} Source ({meta['source']}, page {meta.get('page', 'unknown')}): {doc_text[:800]}"
        )
        citations.append(label + f" {meta['source']}")
    context = "\n\n".join(context_lines)
    # Refined prompt requesting a bullet list
    prompt = (
        "You are a clinical assistant specialised in Parkinson's disease. "
        "Using only the provided context, list all motor symptoms of Parkinson's disease. "
        "Format your answer as a concise bullet list with each symptom on its own line. "
        "If the context does not contain the answer, respond with 'I don't know'.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        max_length=2048,  # allow longer input for larger context
        truncation=True
    )
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens
    )
    raw_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    citation_text = "\n\nSources: " + ", ".join(citations)
    return raw_answer.strip() + citation_text

# Example usage
print(answer_question("What are the motor symptoms of Parkinson's disease?"))

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

slowness and tremor

Sources: [1] doc (14).pdf, [2] doc (15).pdf, [3] doc (14).pdf, [4] doc (2).pdf, [5] doc (14).pdf, [6] doc (14).pdf, [7] doc (12).pdf, [8] doc (6).pdf


In [23]:
print(answer_question(
    "What is the Unified Parkinson’s Disease Rating Scale (UPDRS)?",
    top_k=12,
    max_new_tokens=256
))

I don't know

Sources: [1] doc (13).pdf, [2] doc (1).pdf, [3] doc (6).pdf, [4] doc (14).pdf, [5] doc (12).pdf, [6] doc (10).pdf, [7] doc (9).pdf, [8] doc (11).pdf, [9] doc (12).pdf, [10] doc (15).pdf, [11] doc (11).pdf, [12] doc (14).pdf


In [24]:
docs, metas = retrieve("UPDRS parts", top_k=12)
for meta, doc in zip(metas, docs):
    print(meta['source'], doc[:200])

doc (14).pdf represented among the operated group. Although older 
age is not an absolute exclusion criterion for surgery, 
surgical adverse events are more often encountered in 
this group, l -DOPA- resistant sym
doc (1).pdf 2 = Frequently has numbness, tingling, or aching; not distressing.
1/23/26, 3:27 PM UPDRS - Parkinson’s Disease Research, Education and Clinical Centers
https://www.parkinsons.va.gov/resources/UPDRS.a
doc (1).pdf II. ACTIVITIES OF DAILY LIVING (for both "on" and "off")
1/23/26, 3:27 PM UPDRS - Parkinson’s Disease Research, Education and Clinical Centers
https://www.parkinsons.va.gov/resources/UPDRS.asp 1/8
doc (15).pdf occur in the absence of medication (average im-
provement in off-medication UPDRS III [Unified 
Parkinson’s Disease Rating Scale, part III] score 
of 30 to 50%); it also allows for a reduction in 
med
doc (6).pdf right side, with an anterior-posterior gradient. B: Normal DAT uptake 
in the striatum in a healthy subject.
doc (1).pdf 3 = Food must be 

In [25]:
import gradio as gr

def qa_function(query):
    return answer_question(query, top_k=15)

iface = gr.Interface(
    fn=qa_function,
    inputs=gr.Textbox(lines=2, label="Ask a question about Parkinson's disease"),
    outputs=gr.Textbox(lines=10, label="Answer with citations"),
    title="Parkinson's Disease RAG Assistant"
)
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dd94cb292790028142.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
